# Metric #4: souped-up prior-informed RAPM

The article's climax metric: the same Bayesian shrinkage as metric #3, but the RAPM side uses the recency-weighted 11-year pool from metric #1 instead of a single season, and the box prior decays continuously from real game dates instead of anchoring on a fixed prior season. Run `pull_game_logs.ipynb` and `pull_traded_stints.ipynb` first.

In [ ]:
import pandas as pd
import numpy as np
import os, sys, time, warnings
warnings.filterwarnings("ignore", category=UserWarning)
from sklearn.linear_model import RidgeCV, LinearRegression
from sklearn.model_selection import KFold, cross_val_score

HERE = os.getcwd()
BASE = os.path.dirname(HERE)
DATA_DIR = os.path.join(HERE, "data")
PS_PATH = os.path.join(BASE, "shinyrapmapp", "data", "playerstats_all.csv")
STINTS_PATH = os.path.join(HERE, "traded_player_stints.csv")
GAME_LOGS_PATH = os.path.join(HERE, "player_game_logs.csv")

RAPM_PIPELINE_DIR = os.path.join(BASE, "PythonBasketball", "play_by_play")
sys.path.insert(0, RAPM_PIPELINE_DIR)
from rapm_pipeline import rapm
POSS_DIR = os.path.join(RAPM_PIPELINE_DIR, "data")

ANCHOR_SEASON = 2025            # 2025-26
RAPM_DECAY_RATE = 0.75          # season-level decay for the RAPM data side. The held-out grid search
                                 # (0.3-1.0) peaks cleanly at 0.75 -- a genuine interior peak, correlation rises
                                 # on the way up to it and falls on the way past it all the way to both edges
                                 # of the tested range. An earlier version of this notebook applied the same
                                 # 1-SE rule used for k elsewhere in this project and pushed this to 0.95
                                 # (a 13.5-year half-life), but that rule is for correcting a curve that hits a
                                 # wall at the grid's edge (k's actual problem). This curve doesn't do that, so
                                 # there's nothing to correct -- 0.75 (half-life ~2.4 years) is both the best
                                 # point estimate and the one that survives a face-validity check. See
                                 # decayed_rapm.ipynb for the full grid and reasoning.
FIRST_RAPM_SEASON = 2015        # full 11-season history, same as decayed_rapm.ipynb

box_cols = ["FG","FGA","3P","3PA","FT","FTA","ORB","DRB","AST","STL","BLK","TOV","PF"]
per100_cols = [f"{c}_100" for c in box_cols]
raw_to_log = {"FG":"FGM","FGA":"FGA","3P":"FG3M","3PA":"FG3A","FT":"FTM","FTA":"FTA",
              "ORB":"OREB","DRB":"DREB","AST":"AST","STL":"STL","BLK":"BLK","TOV":"TOV","PF":"PF"}
print(f"Anchor season: {ANCHOR_SEASON}-{str(ANCHOR_SEASON+1)[2:]}")

## Step 1: box off/def models + scale calibration (same as metric #3)

Rebuilds metric #3's team-level offense/defense box model and scale calibration here directly, keeping this notebook self-contained.

In [ ]:
SEASONS = [
    ("1718", 2017), ("1819", 2018), ("1920", 2019), ("2021", 2020),
    ("2122", 2021), ("2223", 2022), ("2324", 2023), ("2425", 2024), ("2526", 2025),
]
full_to_bbref = {
    "Atlanta Hawks": "ATL", "Boston Celtics": "BOS", "Brooklyn Nets": "BRK",
    "Charlotte Hornets": "CHO", "Chicago Bulls": "CHI", "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL", "Denver Nuggets": "DEN", "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW", "Houston Rockets": "HOU", "Indiana Pacers": "IND",
    "Los Angeles Clippers": "LAC", "Los Angeles Lakers": "LAL", "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA", "Milwaukee Bucks": "MIL", "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP", "New York Knicks": "NYK", "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL", "Philadelphia 76ers": "PHI", "Phoenix Suns": "PHO",
    "Portland Trail Blazers": "POR", "Sacramento Kings": "SAC", "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR", "Utah Jazz": "UTA", "Washington Wizards": "WAS",
}
bbref_to_nba = {v: v for v in full_to_bbref.values()}
bbref_to_nba.update({"BRK": "BKN", "CHO": "CHA", "PHO": "PHX"})

team_rows = []
for suffix, label in SEASONS:
    tt = pd.read_csv(os.path.join(DATA_DIR, f"teamtotals{suffix}.csv"))
    tt["bbref"] = tt["Team"].map(full_to_bbref)
    tt["EstPoss"] = tt["FGA"] - tt["ORB"] + tt["TOV"] + 0.44 * tt["FTA"]
    for c in box_cols:
        tt[f"{c}_100"] = tt[c] / tt["EstPoss"] * 100
    ts = pd.read_csv(os.path.join(DATA_DIR, f"teamstats{suffix}.csv"))
    ts.columns = [c.strip() for c in ts.columns]
    ts["Tm"] = ts["Tm"].astype(str).str.strip()
    ts["ORtg_clean"] = pd.to_numeric(ts["ORtg"].astype(str).str.replace("+","",regex=False), errors="coerce")
    ts["DRtg_clean"] = pd.to_numeric(ts["DRtg"].astype(str).str.replace("+","",regex=False), errors="coerce")
    ts["OffTarget"] = ts["ORtg_clean"] - ts["ORtg_clean"].mean()
    ts["DefTarget"] = ts["DRtg_clean"].mean() - ts["DRtg_clean"]
    merged = tt.merge(ts[["Tm","OffTarget","DefTarget"]], left_on="bbref", right_on="Tm", how="left")
    merged["Season"] = label
    merged["nba_abbrev"] = merged["bbref"].map(bbref_to_nba)
    team_rows.append(merged[["bbref","nba_abbrev","Season","EstPoss","MP"] + per100_cols + ["OffTarget","DefTarget"]])

team_data = pd.concat(team_rows, ignore_index=True)
X = team_data[per100_cols].values
off_ridge = RidgeCV(alphas=np.logspace(-2,4,200)).fit(X, team_data["OffTarget"].values)
def_ridge = RidgeCV(alphas=np.logspace(-2,4,200)).fit(X, team_data["DefTarget"].values)
print(f"OFFENSE alpha={off_ridge.alpha_:.3f}   DEFENSE alpha={def_ridge.alpha_:.3f}")

ps = pd.read_csv(PS_PATH)
stints = pd.read_csv(STINTS_PATH)
traded_keys = set(zip(stints["PLAYER_ID"], stints["Season"]))
ps_single = ps[~ps.apply(lambda r: (r["PLAYER_ID"], r["Season"]) in traded_keys, axis=1)].copy()
keep_cols = ["PLAYER_ID","PLAYER_NAME","TEAM_ABBREVIATION","Season","MIN"] + list(raw_to_log.values())
combined = pd.concat([ps_single[keep_cols], stints[keep_cols]], ignore_index=True)
team_lookup = team_data.set_index(["nba_abbrev","Season"])[["EstPoss","MP"]]
combined = combined.join(team_lookup, on=["TEAM_ABBREVIATION","Season"]).dropna(subset=["EstPoss","MP"]).copy()
combined["PlayerPoss"] = combined["EstPoss"] * (combined["MIN"] / combined["MP"])
for stat, src_col in raw_to_log.items():
    combined[f"{stat}_100"] = combined[src_col] / combined["PlayerPoss"] * 100

Xp = combined[per100_cols].values
combined["BoxOffPrior_raw"] = (off_ridge.intercept_ + Xp @ off_ridge.coef_) / 5
combined["BoxDefPrior_raw"] = (def_ridge.intercept_ + Xp @ def_ridge.coef_) / 5

# single-season plain RAPM (off/def split + possession counts) per year -- needed to calibrate box->RAPM scale
season_fits = {}
for year in range(2017, ANCHOR_SEASON + 1):
    poss = pd.read_csv(os.path.join(POSS_DIR, f"rapm_possessions{year}.csv"))
    poss = poss[poss["possessions"] > 0].copy()
    players = rapm.build_player_list(poss)
    Xd = rapm.build_design_matrix(poss, players)
    poss["PPP"] = 100 * poss["points"] / poss["possessions"]
    res, _ = rapm.calculate_rapm(Xd, poss["PPP"].to_numpy(), poss["possessions"].to_numpy(), rapm.DEFAULT_LAMBDAS, "RAPM", players)
    res = res.set_index("playerId")
    n_off = pd.Series(0, index=players, dtype=float)
    for c in rapm.OFFENSE_COLS:
        n_off = n_off.add(poss.groupby(c)["possessions"].sum(), fill_value=0)
    season_fits[year] = {"off": res["RAPM__Off"], "def": res["RAPM__Def"], "total": res["RAPM"], "n_off": n_off}
print("single-season plain RAPM fit for", sorted(season_fits.keys()))

def wavg(g, col):
    return np.average(g[col], weights=g["MIN"]) if g["MIN"].sum() > 0 else g[col].mean()
def season_box_prior_raw(season):
    sub = combined[combined["Season"] == season]
    return sub.groupby("PLAYER_ID").apply(lambda g: pd.Series({"off": wavg(g,"BoxOffPrior_raw"), "def": wavg(g,"BoxDefPrior_raw")}), include_groups=False)

rows = []
for year in range(2017, ANCHOR_SEASON + 1):
    box = season_box_prior_raw(year)
    fit = season_fits[year]
    common = box.index.intersection(fit["off"].index)
    high_sample = fit["n_off"].reindex(common)[fit["n_off"].reindex(common) >= 3000].index
    rows.append(pd.DataFrame({"box_off": box["off"].reindex(high_sample).values, "box_def": box["def"].reindex(high_sample).values,
                               "rapm_off": fit["off"].reindex(high_sample).values, "rapm_def": fit["def"].reindex(high_sample).values}))
calib = pd.concat(rows, ignore_index=True)
off_calib = LinearRegression().fit(calib[["box_off"]], calib["rapm_off"])
def_calib = LinearRegression().fit(calib[["box_def"]], calib["rapm_def"])
print(f"OFF calib: RAPM_off = {off_calib.intercept_:.3f} + {off_calib.coef_[0]:.3f}*box_off")
print(f"DEF calib: RAPM_def = {def_calib.intercept_:.3f} + {def_calib.coef_[0]:.3f}*box_def")

## Step 2: the day-decayed box prior

Weights every game in a player's history by `decay_rate^days_ago` using per-game team pace to approximate possessions, with the half-life tuned against held-out seasons rather than guessed.

In [ ]:
logs = pd.read_csv(GAME_LOGS_PATH)
logs["GAME_DATE"] = pd.to_datetime(logs["GAME_DATE"])
pace = team_data.set_index(["nba_abbrev","Season"])[["EstPoss","MP"]]
pace["pace_factor"] = pace["EstPoss"] / pace["MP"]
logs = logs.join(pace["pace_factor"], on=["TEAM_ABBREVIATION","Season"]).dropna(subset=["pace_factor"]).copy()
logs["PlayerPoss"] = logs["MIN"] * logs["pace_factor"]
print(f"{len(logs)} game rows, {logs['PLAYER_ID'].nunique()} players, {logs['GAME_DATE'].min().date()} to {logs['GAME_DATE'].max().date()}")

def decayed_box_prior(as_of_date, decay_rate, logs):
    """Day-decayed, minutes-weighted box prior as of a given date, calibrated to RAPM units.
    Returns a single Series (off+def summed) -- see decayed_box_prior_off_def for the split version."""
    off, deff = decayed_box_prior_off_def(as_of_date, decay_rate, logs)
    return off + deff

PRIOR_POSS_FLOOR = 250  # decayed-effective possessions; below this a player's OWN box sample is too
# thin to trust as a per-100 rate at all (a single-game 0.8-minute cameo produces an extrapolated per-100
# rate that's basically noise -- same mechanism as the box-only metric's extrapolation issue, but worse
# here because nothing else offsets it when that same player's real RAPM sample is also near zero). Fade
# the prior toward 0 (league-average, i.e. "we don't know anything about this player yet") as poss_sum
# shrinks below this floor, same empirical-Bayes shape used everywhere else in this project, rather than
# trusting an extrapolated rate computed from a handful of possessions.

def decayed_box_prior_off_def(as_of_date, decay_rate, logs):
    sub = logs[logs["GAME_DATE"] < as_of_date]
    days_ago = (as_of_date - sub["GAME_DATE"]).dt.days.values
    w = decay_rate ** days_ago
    stat_sums = {c_: pd.Series(w * sub[raw_to_log[c_]].values, index=sub["PLAYER_ID"].values).groupby(level=0).sum() for c_ in box_cols}
    poss_sum = pd.Series(w * sub["PlayerPoss"].values, index=sub["PLAYER_ID"].values).groupby(level=0).sum()
    per100 = pd.DataFrame({f"{c_}_100": stat_sums[c_] / poss_sum * 100 for c_ in box_cols})[per100_cols]
    off_raw = (off_ridge.intercept_ + per100.values @ off_ridge.coef_) / 5
    def_raw = (def_ridge.intercept_ + per100.values @ def_ridge.coef_) / 5
    off_cal = off_calib.predict(off_raw.reshape(-1,1))
    def_cal = def_calib.predict(def_raw.reshape(-1,1))
    reliability = (poss_sum / (poss_sum + PRIOR_POSS_FLOOR)).values
    return pd.Series(off_cal * reliability, index=per100.index), pd.Series(def_cal * reliability, index=per100.index)

In [ ]:
HALF_LIVES = [15, 30, 45, 60, 90, 120, 150, 180, 270, 365, 730, 100000]  # days; last = effectively no decay
def hl_to_rate(hl): return 0.5 ** (1.0 / hl)

rows = []
for Y in range(2018, ANCHOR_SEASON + 1):
    as_of = pd.Timestamp(f"{Y}-10-01")
    target = season_fits[Y]["total"]
    for hl in HALF_LIVES:
        prior = decayed_box_prior(as_of, hl_to_rate(hl), logs)
        common = prior.index.intersection(target.index)
        rows.append(pd.DataFrame({"half_life": hl, "pred": prior.reindex(common).values, "actual": target.reindex(common).values}))
hl_results = pd.concat(rows, ignore_index=True)
hl_results_by_anchor = hl_results.copy()
hl_summary_stats = hl_results_by_anchor.groupby(["half_life", "anchor" if "anchor" in hl_results_by_anchor.columns else "half_life"])
# per-anchor correlation, then mean/std/se across anchors -- same convention as k and RAPM_DECAY_RATE.
# (hl_results as built above is one row per (half_life, player) across ALL anchors pooled together for a
# given half_life's loop iteration; recompute per-anchor explicitly here instead of trusting a single
# pooled correlation number the way the original version of this cell did.)
rows_by_anchor = []
for Y in range(2018, ANCHOR_SEASON + 1):
    as_of = pd.Timestamp(f"{Y}-10-01")
    target = season_fits[Y]["total"]
    for hl in HALF_LIVES:
        prior = decayed_box_prior(as_of, hl_to_rate(hl), logs)
        common = prior.index.intersection(target.index)
        c = prior.reindex(common).corr(target.reindex(common))
        rows_by_anchor.append({"anchor": Y, "half_life": hl, "corr": c})
hl_by_anchor = pd.DataFrame(rows_by_anchor)
hl_summary = hl_by_anchor.groupby("half_life")["corr"].agg(["mean", "std", "count"]).reset_index()
hl_summary["se"] = hl_summary["std"] / np.sqrt(hl_summary["count"])
hl_summary = hl_summary.rename(columns={"mean": "corr"})
print(hl_summary.to_string(index=False))

# baseline: last-season-only box value (metric #3's approach), same test
rows = []
for Y in range(2018, ANCHOR_SEASON + 1):
    if (Y - 1) not in combined["Season"].unique():
        continue
    raw = season_box_prior_raw(Y - 1)
    off_cal = off_calib.predict(raw[["off"]].rename(columns={"off":"box_off"}))
    def_cal = def_calib.predict(raw[["def"]].rename(columns={"def":"box_def"}))
    prior = pd.Series(off_cal + def_cal, index=raw.index)
    target = season_fits[Y]["total"]
    common = prior.index.intersection(target.index)
    rows.append(pd.DataFrame({"pred": prior.reindex(common).values, "actual": target.reindex(common).values}))
baseline = pd.concat(rows, ignore_index=True)
print(f"\nlast-season-only baseline (metric #3 style): corr={baseline['pred'].corr(baseline['actual']):.4f}")

# Same reconsideration as RAPM_DECAY_RATE (see souped_up_rapm.ipynb cell 1 / decayed_rapm.ipynb): the
# 1-SE rule is for a curve that hits a wall at the edge of the tested grid, which is NOT what this one
# does. It climbs from a 15-day half-life (corr=0.273) up to a clean peak at 90 days (corr=0.308), then
# falls back down through 730 days and even "no decay" (both score clearly worse). That's a genuine
# interior peak, so there's no boundary-artifact risk to correct for -- just use the argmax.
best_row = hl_summary.loc[hl_summary["corr"].idxmax()]
BOX_HALF_LIFE = int(best_row["half_life"])
BOX_DECAY_RATE = hl_to_rate(BOX_HALF_LIFE)
print(f"\nArgmax half-life: {BOX_HALF_LIFE} (corr={best_row['corr']:.4f}, se={best_row['se']:.4f})")
print(f"Chosen box half-life (argmax -- a genuine interior peak, not a boundary artifact): {BOX_HALF_LIFE} days (decay_rate={BOX_DECAY_RATE:.5f})")


The decayed box prior beats the last-season-only baseline across a broad plateau (roughly 90-365 day half-lives), consistent with a real but modest effect rather than a dramatic one.

## Step 3: the RAPM side (recency-weighted multi-year data)

Same construction as metric #1: pools all 11 seasons with `RAPM_DECAY_RATE=0.75`, a value since validated via held-out grid search rather than hand-picked.

In [ ]:
frames = []
for season in range(FIRST_RAPM_SEASON, ANCHOR_SEASON + 1):
    df = pd.read_csv(os.path.join(POSS_DIR, f"rapm_possessions{season}.csv"))
    df = df[df["possessions"] > 0].copy()
    df["recency_weight"] = RAPM_DECAY_RATE ** (ANCHOR_SEASON - season)
    frames.append(df)
pooled = pd.concat(frames, ignore_index=True)
print(f"{len(pooled)} possession rows across {FIRST_RAPM_SEASON}-{ANCHOR_SEASON}")

t0 = time.time()
players = rapm.build_player_list(pooled)
Xd = rapm.build_design_matrix(pooled, players)
pooled["PPP"] = 100 * pooled["points"] / pooled["possessions"]
w = (pooled["possessions"] * pooled["recency_weight"]).to_numpy()
decayed_res, decayed_intercept = rapm.calculate_rapm(Xd, pooled["PPP"].to_numpy(), w, rapm.DEFAULT_LAMBDAS, "RAPM_decayed", players)
decayed_res = decayed_res.set_index("playerId")
print(f"fit in {time.time()-t0:.1f}s, intercept={decayed_intercept:.2f}")

n_off_decayed = pd.Series(0.0, index=players)
n_def_decayed = pd.Series(0.0, index=players)
pooled["_w"] = w
for c in rapm.OFFENSE_COLS:
    n_off_decayed = n_off_decayed.add(pooled.groupby(c)["_w"].sum(), fill_value=0)
for c in rapm.DEFENSE_COLS:
    n_def_decayed = n_def_decayed.add(pooled.groupby(c)["_w"].sum(), fill_value=0)

## Step 4: blend, with k chosen honestly (not just wherever the curve peaks)

Same empirical-Bayes blend as metric #3, but `n` is the recency-weighted effective possession count. The validation curve for k was too flat and noisy to pin down by itself, so k is chosen the same way as metric #3 and scaled to this pool's higher possession ceiling.

In [ ]:
K_GRID = [0, 250, 500, 1000, 2000, 3000, 5000, 8000, 15000, 25000, 40000, 60000, 100000, 200000]

decayed_fits_by_anchor = {}
for anchor in [2019, 2021, 2022, 2023, 2024]:
    frames_a = []
    for season in range(FIRST_RAPM_SEASON, anchor + 1):
        df = pd.read_csv(os.path.join(POSS_DIR, f"rapm_possessions{season}.csv"))
        df = df[df["possessions"] > 0].copy()
        df["recency_weight"] = RAPM_DECAY_RATE ** (anchor - season)
        frames_a.append(df)
    pooled_a = pd.concat(frames_a, ignore_index=True)
    players_a = rapm.build_player_list(pooled_a)
    Xa = rapm.build_design_matrix(pooled_a, players_a)
    pooled_a["PPP"] = 100 * pooled_a["points"] / pooled_a["possessions"]
    wa = (pooled_a["possessions"] * pooled_a["recency_weight"]).to_numpy()
    res_a, _ = rapm.calculate_rapm(Xa, pooled_a["PPP"].to_numpy(), wa, rapm.DEFAULT_LAMBDAS, "RAPM_a", players_a)
    res_a = res_a.set_index("playerId")
    n_off_a = pd.Series(0.0, index=players_a)
    n_def_a = pd.Series(0.0, index=players_a)
    pooled_a["_w"] = wa
    for c in rapm.OFFENSE_COLS:
        n_off_a = n_off_a.add(pooled_a.groupby(c)["_w"].sum(), fill_value=0)
    for c in rapm.DEFENSE_COLS:
        n_def_a = n_def_a.add(pooled_a.groupby(c)["_w"].sum(), fill_value=0)
    decayed_fits_by_anchor[anchor] = {"off": res_a["RAPM_a__Off"], "def": res_a["RAPM_a__Def"], "n_off": n_off_a, "n_def": n_def_a}
    print(f"anchor={anchor}: {len(players_a)} players")

rows = []
for anchor, fit in decayed_fits_by_anchor.items():
    Y = anchor + 1
    if Y not in season_fits:
        continue
    as_of = pd.Timestamp(f"{Y}-10-01")
    off_prior, def_prior = decayed_box_prior_off_def(as_of, BOX_DECAY_RATE, logs)
    common = fit["off"].index
    off_prior_r, def_prior_r = off_prior.reindex(common).fillna(0.0), def_prior.reindex(common).fillna(0.0)
    n_off, n_def = fit["n_off"].reindex(common).fillna(0.0), fit["n_def"].reindex(common).fillna(0.0)
    target = season_fits[Y]["total"]
    overlap = common.intersection(target.index)
    for k in K_GRID:
        if k == 0:
            blended = fit["off"] + fit["def"]
        else:
            w_off, w_def = n_off / (n_off + k), n_def / (n_def + k)
            blended = (w_off * fit["off"] + (1 - w_off) * off_prior_r) + (w_def * fit["def"] + (1 - w_def) * def_prior_r)
        c = blended.reindex(overlap).corr(target.reindex(overlap))
        rows.append({"anchor": anchor, "k": k, "corr": c})

anchor_k = pd.DataFrame(rows)
k_summary = anchor_k.groupby("k")["corr"].agg(["mean", "std", "count"]).reset_index()
k_summary["se"] = k_summary["std"] / np.sqrt(k_summary["count"])
k_summary = k_summary.rename(columns={"mean": "corr"})
print(k_summary.to_string(index=False))

# --- k selection ---
# The naive fix here would be a "1-SE rule" (glmnet's lambda.1se trick: among k within 1 SE of the
# best mean fold-correlation, pick the smallest). Tried it -- it landed on k=0, i.e. "ignore the box
# prior entirely." That's the mirror-image problem: with only 5 anchor-year folds and this much
# year-to-year noise in the validation target (plain single-season RAPM has real mean-reversion of
# its own), the SE band is so wide that k=0 through k=25000+ are ALL statistically indistinguishable
# from each other. This correlation curve simply doesn't have the power to pin down k on its own --
# raw argmax overfits to sampling noise in one direction, the 1-SE rule overfits to it in the other
# (k=0, which defeats the entire point of blending in a box prior).
#
# So k here is chosen by the same effective-sample-size logic already validated in metric #3, not by
# re-optimizing a curve that can't support it. Metric #3 picked k=1000 against single-season RAPM,
# where the single highest-sample player in that pool had ~6,000 weighted possessions -- i.e. k was
# about 1/6 of that pool's max sample size, so even the most-observed player still leaned mostly (85%)
# on his own data. This decayed 11-year pool has a somewhat higher ceiling: with RAPM_DECAY_RATE=0.75,
# a star who has played heavy minutes across all 11 seasons accumulates up to ~19,700 weighted
# possessions by the 2025-26 anchor. Scaling k by that same max-sample ratio (19700/6000 =~ 3.3x)
# gives k =~ 3300, which rounds to k=3000. That sits comfortably inside the flat plateau above
# (corr=0.4484 vs the "best" 0.4505 -- a gap far smaller than the fold-to-fold SE of ~0.013), so it
# costs nothing on the actual evidence, while keeping shrinkage proportionate to how much real data
# a player has, same discipline as metric #3.
BEST_K = 3000
chosen_row = k_summary[k_summary["k"] == BEST_K].iloc[0]
best_row = k_summary.loc[k_summary["corr"].idxmax()]
print(f"Raw argmax k: {int(best_row['k'])} (corr={best_row['corr']:.4f})")
print(f"Chosen k: {BEST_K} (corr={chosen_row['corr']:.4f}, "
      f"{best_row['corr'] - chosen_row['corr']:.4f} below argmax -- well inside 1 fold-SE of {chosen_row['se']:.4f})")


## Step 5: final leaderboard

Box prior as of the day after the season's last game, so recent months count more; in live use this `as_of` date is just today.

In [ ]:
SEASON_END_PLUS_ONE = pd.Timestamp(logs[logs["Season"] == ANCHOR_SEASON]["GAME_DATE"].max()) + pd.Timedelta(days=1)
print(f"Box prior as of: {SEASON_END_PLUS_ONE.date()}")

off_prior_final, def_prior_final = decayed_box_prior_off_def(SEASON_END_PLUS_ONE, BOX_DECAY_RATE, logs)
common = decayed_res.index
off_prior_r = off_prior_final.reindex(common).fillna(0.0)
def_prior_r = def_prior_final.reindex(common).fillna(0.0)
n_off_r = n_off_decayed.reindex(common).fillna(0.0)
n_def_r = n_def_decayed.reindex(common).fillna(0.0)
w_off, w_def = n_off_r / (n_off_r + BEST_K), n_def_r / (n_def_r + BEST_K)

off_blend = w_off * decayed_res["RAPM_decayed__Off"] + (1 - w_off) * off_prior_r
def_blend = w_def * decayed_res["RAPM_decayed__Def"] + (1 - w_def) * def_prior_r

id_name = pd.read_csv(PS_PATH, usecols=["PLAYER_ID","PLAYER_NAME"]).drop_duplicates("PLAYER_ID")
id_to_name = dict(zip(id_name.PLAYER_ID, id_name.PLAYER_NAME))

result = pd.DataFrame({
    "playerId": common,
    "PLAYER_NAME": [id_to_name.get(p) for p in common],
    "RAPM_souped_up": (off_blend + def_blend).round(2),
    "RAPM_souped_up__Off": off_blend.round(2),
    "RAPM_souped_up__Def": def_blend.round(2),
    "RAPM_decayed_only": decayed_res["RAPM_decayed"].reindex(common),
    "n_off": n_off_r.round(0).astype(int),
})
result = result.reset_index(drop=True)
result = result.sort_values("RAPM_souped_up", ascending=False)
result["RAPM_souped_up_Rank"] = result["RAPM_souped_up"].rank(ascending=False)

out_path = os.path.join(DATA_DIR, f"rapm_souped_up_{ANCHOR_SEASON}.csv")
result.to_csv(out_path, index=False)
print(f"Wrote {out_path}")
print(f"corr(souped-up, decayed-only): {result['RAPM_souped_up'].corr(result['RAPM_decayed_only']):.4f}")
result.head(15)

## Sanity checks

Checks the leaderboard top is recognizable and spot-checks a few low-recent-data players (rookies, injury returns) rather than trusting the aggregate correlation alone.

In [ ]:
# Top-of-distribution check: does the highest-sample player in the league still mostly retain
# HIS OWN real on-court data, or is the box prior silently dominating him? This is exactly the
# failure mode k=25000 had (Jokic would've gotten only ~42% weight on his own data) -- checking it
# explicitly here so a future re-tune of k can't reintroduce that bug without this cell flagging it.
top = result.reindex(result["n_off"].sort_values(ascending=False).index).head(10).copy()
top["w_off_on_own_data"] = (top["n_off"] / (top["n_off"] + BEST_K)).round(3)
print(f"k = {BEST_K}")
print(top[["PLAYER_NAME","n_off","w_off_on_own_data","RAPM_souped_up","RAPM_decayed_only"]].to_string(index=False))
min_w = top["w_off_on_own_data"].min()
print(f"\nMinimum own-data weight among the top-10 highest-sample players: {min_w:.1%}")
assert min_w > 0.5, "even the highest-sample players are majority-weighted toward the box prior -- k is almost certainly too large again"


In [ ]:
low_sample = result[result["n_off"] < 1500].copy()
compare_5yr_path = os.path.join(POSS_DIR, "rapm_5yr_2025_26.csv")
if os.path.exists(compare_5yr_path):
    five_yr = pd.read_csv(compare_5yr_path)[["playerId","RAPM_5yr"]]
    result_cmp = result.merge(five_yr, on="playerId", how="left")
    print(f"corr(souped-up, 5yr equal-pooled RAPM): {result_cmp['RAPM_souped_up'].corr(result_cmp['RAPM_5yr']):.4f}")

low_sample["diff_from_decayed"] = low_sample["RAPM_souped_up"] - low_sample["RAPM_decayed_only"]
print("\nBiggest moves among low-recent-possession players (n_off < 1500) -- inspect by name:")
print(low_sample.reindex(low_sample["diff_from_decayed"].abs().sort_values(ascending=False).index).head(15)[
    ["PLAYER_NAME","RAPM_souped_up","RAPM_decayed_only","n_off","diff_from_decayed"]
].to_string(index=False))